# 🌊 ML — Aprendizaje No Supervisado
## Laboratorio Completo: Dataset Sintético + Sea Animals Image Dataset
### Materia: Machine Learning | Aprendizaje No Supervisado

---
> Ejecuta las celdas de arriba hacia abajo, en orden. No saltes secciones.
> Antes de la Sección 2 sube el dataset a Google Drive en la ruta indicada.


---
## 🔧 Sección 0 — Configuración del Entorno


In [ ]:
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import os, random
np.random.seed(42)
random.seed(42)
print('✅ Librerias base importadas correctamente')


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('✅ Google Drive montado')


### 📁 Estructura de carpetas requerida en Google Drive

```
MyDrive/
└── ML_Lab_NoSupervisado/
    └── Sea Animals/
        ├── Clams/
        ├── Corals/
        ├── Crab/
        ├── Dolphin/
        ├── Eel/
        ├── Fish/
        ├── Jelly Fish/
        ├── Lobster/
        ├── Nudibranchs/
        ├── Octopus/
        ├── Penguin/
        ├── Puffers/
        ├── Rays/
        ├── Seahorse/
        ├── Sea Otter/
        ├── Sea Urchins/
        ├── Seal/
        ├── Shark/
        ├── Shrimp/
        ├── Squid/
        ├── Starfish/
        ├── Turtle/
        └── Whale/
```

> Descarga desde: https://www.kaggle.com/datasets/vencerlanz09/sea-animals-image-dataste


In [ ]:
RUTA_BASE = '/content/drive/MyDrive/ML_Lab_NoSupervisado/Sea Animals'
CLASES_ESPERADAS = [
    'Clams','Corals','Crab','Dolphin','Eel','Fish',
    'Jelly Fish','Lobster','Nudibranchs','Octopus','Penguin',
    'Puffers','Rays','Seahorse','Sea Otter','Sea Urchins',
    'Seal','Shark','Shrimp','Squid','Starfish','Turtle','Whale'
]
if os.path.exists(RUTA_BASE):
    print(f'✅ Ruta base encontrada: {RUTA_BASE}')
    for c in CLASES_ESPERADAS:
        ruta_c = os.path.join(RUTA_BASE, c)
        if os.path.isdir(ruta_c):
            n = len([f for f in os.listdir(ruta_c) if f.lower().endswith(('.jpg','.jpeg','.png','.webp'))])
            print(f'  ✅ {c}: {n} imagenes')
        else:
            print(f'  ❌ {c}: carpeta NO encontrada')
else:
    print(f'❌ Ruta base NO encontrada: {RUTA_BASE}')
    print('   Crea la estructura de carpetas descrita arriba.')


---
## 📊 Sección 1 — Punto 1: Dataset Sintético + KMeans

Trabajamos sobre datos **generados artificialmente** con `make_blobs`.
Objetivo: aprender KMeans antes de aplicarlo a datos reales.

> **Conceptos:** clustering, centroides, hard/soft clustering, Método del Codo, Silhouette Score.


### 1.1 — Generación del Dataset Sintético

Generamos entre **1 y 20 centroides** aleatoriamente, con distancia mínima importante entre ellos para que los clusters sean visualmente separables.


In [ ]:
from sklearn.datasets import make_blobs
import numpy as np, random

# {#yo: make_blobs genera clusters sintéticos. 'centers' = coordenadas de centroides,
#       'cluster_std' = dispersión de puntos alrededor de cada centroide,
#       'n_samples' = total de puntos. Es la base para validar que KMeans
#       recupera los clusters que nosotros definimos artificialmente.}

# Paso 1: número aleatorio de centroides entre 1 y 20
n_centers = random.randint(1, 20)
print(f'🎲 Numero de clusters generado aleatoriamente: {n_centers}')

# {#yo: Distancia mínima entre centroides = garantía de separabilidad visual.
#       np.linalg.norm() calcula la distancia euclidiana entre dos puntos.
#       Si la distancia candidato-centros_ya_aceptados > DISTANCIA_MINIMA, lo aceptamos.
#       DISTANCIA_MINIMA = 5.0 -> clusters bien separados en la gráfica.}
DISTANCIA_MINIMA = 5.0
blob_centers = []
intentos = 0
while len(blob_centers) < n_centers:
    candidato = np.random.uniform(-15, 15, size=2)
    if all(np.linalg.norm(candidato - c) > DISTANCIA_MINIMA for c in blob_centers):
        blob_centers.append(candidato)
    intentos += 1
    if intentos > 50000:
        print('⚠️  Muchos intentos, reduciendo distancia minima a 3.0')
        DISTANCIA_MINIMA = 3.0
        intentos = 0

blob_centers = np.array(blob_centers)

# {#yo: cluster_std pequeño (0.6) = puntos muy compactos alrededor del centroide.
#       Valor grande = clusters dispersos y solapados. Elegimos 0.6 para separabilidad.}
blob_std = np.full(n_centers, 0.6)

X, y = make_blobs(n_samples=2000, centers=blob_centers, cluster_std=blob_std, random_state=42)

print(f'✅ Dataset generado: {X.shape[0]} muestras, {X.shape[1]} features')
print(f'📍 Coordenadas de los {n_centers} centroides:')
for i, c in enumerate(blob_centers):
    print(f'   Cluster {i}: ({c[0]:.2f}, {c[1]:.2f})')


### 1.2 — Visualización del Dataset Sintético


In [ ]:
# Función ORIGINAL del docente — preservada íntegra
def plot_clusters(X, y=None):
    plt.scatter(X[:, 0], X[:, 1], c=y, s=1)
    plt.xlabel('$x_1$', fontsize=14)
    plt.ylabel('$x_2$', fontsize=14, rotation=0)

plt.figure(figsize=(9, 5))
plot_clusters(X, y)
plt.title(f'Dataset Sintético — {n_centers} clusters generados aleatoriamente', fontsize=14)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print(f'📊 {n_centers} clusters, {X.shape[0]} puntos totales')


### 1.3 — Entrenamiento de KMeans


In [ ]:
from sklearn.cluster import KMeans

# {#yo: Parámetros críticos de KMeans:
#   n_clusters : número de grupos k a encontrar (el hiperparámetro más importante)
#   init       : 'k-means++' = inicialización inteligente (distribuye centroides lejos entre sí)
#                'random'    = inicialización aleatoria (puede dar resultados malos)
#   n_init     : cuántas veces se reinicializa. Se queda con el de menor inercia.
#   algorithm  : 'lloyd' = KMeans clásico | 'elkan' = usa desigualdad triangular (más rápido)
#   random_state=42 : garantiza reproducibilidad (mismos resultados siempre)}

k = n_centers  # usamos exactamente el número de clusters generados
kmeans = KMeans(n_clusters=k, random_state=42)
y_pred = kmeans.fit_predict(X)

print(f'✅ KMeans entrenado con k={k}')
print('📍 Centroides encontrados:')
print(kmeans.cluster_centers_)
print('\n📊 Distribución por cluster:')
unique, counts = np.unique(y_pred, return_counts=True)
for cl, cnt in zip(unique, counts):
    print(f'   Cluster {cl}: {cnt} muestras')


### 1.4 — Funciones de Visualización del Docente

Preservadas **íntegramente** del cuadernillo original.


In [ ]:
# ── Funciones del docente (preservadas íntegras) ───────────────────────────
def plot_data(X):
    plt.plot(X[:, 0], X[:, 1], 'k.', markersize=2)

def plot_centroids(centroids, weights=None, circle_color='w', cross_color='k'):
    if weights is not None:
        centroids = centroids[weights > weights.max() / 10]
    plt.scatter(centroids[:, 0], centroids[:, 1],
                marker='o', s=10, linewidths=8,
                color=circle_color, zorder=10, alpha=0.9)
    plt.scatter(centroids[:, 0], centroids[:, 1],
                marker='x', s=2, linewidths=10,
                color=cross_color, zorder=11, alpha=1)

def plot_decision_boundaries(clusterer, X, resolution=1000, show_centroids=True,
                             show_xlabels=True, show_ylabels=True):
    mins = X.min(axis=0) - 0.1
    maxs = X.max(axis=0) + 0.1
    xx, yy = np.meshgrid(np.linspace(mins[0], maxs[0], resolution),
                         np.linspace(mins[1], maxs[1], resolution))
    Z = clusterer.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    plt.contourf(Z, extent=(mins[0], maxs[0], mins[1], maxs[1]), cmap='Pastel2')
    plt.contour(Z, extent=(mins[0], maxs[0], mins[1], maxs[1]),
                linewidths=1, colors='k')
    plot_data(X)
    if show_centroids:
        plot_centroids(clusterer.cluster_centers_)
    if show_xlabels:
        plt.xlabel('$x_1$', fontsize=14)
    else:
        plt.tick_params(labelbottom=False)
    if show_ylabels:
        plt.ylabel('$x_2$', fontsize=14, rotation=0)
    else:
        plt.tick_params(labelleft=False)

print('✅ Funciones de visualización del docente cargadas')


### 1.5 — Clusters y Fronteras de Decisión


In [ ]:
plt.figure(figsize=(9, 5))
plot_decision_boundaries(kmeans, X)
plt.title(f'KMeans — Fronteras de decisión ({k} clusters)', fontsize=14)
plt.tight_layout()
plt.show()


### 1.6 — Hard Clustering vs Soft Clustering

**Hard clustering:** cada muestra recibe UNA etiqueta discreta (`labels_`).

**Soft clustering:** cada muestra recibe una distancia a cada centroide (`transform()`), que indica cuán segura es la asignación.


In [ ]:
# {#yo: HARD CLUSTERING — kmeans.labels_
#   Devuelve un entero por muestra: a qué cluster pertenece (0 a k-1).
#   Asignación binaria: la muestra pertenece AL cluster más cercano y solo a ese.

#   SOFT CLUSTERING — kmeans.transform(X)
#   Devuelve matriz (m, k): distancia euclidiana de cada muestra a cada centroide.
#   Distancia pequeña = muestra muy cerca del centroide = asignación segura.
#   Útil para detectar muestras ambiguas (cerca de fronteras de decisión).}

print('=' * 55)
print('HARD CLUSTERING — kmeans.labels_')
print('=' * 55)
print(f'Shape: {kmeans.labels_.shape}')
print(f'Primeras 20 etiquetas: {kmeans.labels_[:20]}')

print('\n' + '=' * 55)
print('SOFT CLUSTERING — kmeans.transform()')
print('=' * 55)
X_new = np.array([[0, 2], [3, 2], [-3, 3], [-3, 2.5]])
distancias = kmeans.transform(X_new)
print(f'Forma de la matriz de distancias: {distancias.shape}  (muestras x clusters)')
for i, d in enumerate(distancias):
    cl = np.argmin(d)
    print(f'  Punto {i}: cluster={cl} | dist_min={d[cl]:.4f}')


### 1.7 — Proceso Iterativo de KMeans

Visualizamos la convergencia del algoritmo iteración a iteración.


In [ ]:
# {#yo: KMeans es iterativo:
#   1. Inicializa centroides (random o k-means++)
#   2. Asigna cada punto al centroide más cercano (regiones de Voronoi)
#   3. Recalcula centroides como promedio de sus puntos asignados
#   4. Repite hasta convergencia (cambio < tol) o max_iter
#   Usamos max_iter=1,2,3 para ver la evolución visual paso a paso.}

kmeans_iter1 = KMeans(n_clusters=k, init='k-means++', n_init=1,
                      algorithm='elkan', max_iter=1, random_state=4)
kmeans_iter2 = KMeans(n_clusters=k, init='k-means++', n_init=1,
                      algorithm='elkan', max_iter=2, random_state=4)
kmeans_iter3 = KMeans(n_clusters=k, init='k-means++', n_init=1,
                      algorithm='elkan', max_iter=3, random_state=4)
kmeans_iter1.fit(X)
kmeans_iter2.fit(X)
kmeans_iter3.fit(X)

plt.figure(figsize=(12, 8))
plt.subplot(321)
plot_data(X)
plot_centroids(kmeans_iter1.cluster_centers_, circle_color='r', cross_color='w')
plt.ylabel('$x_2$', fontsize=14, rotation=0)
plt.tick_params(labelbottom=False)
plt.title('Centroides iniciales (iter 1)', fontsize=11)
plt.subplot(322)
plot_decision_boundaries(kmeans_iter1, X, show_xlabels=False, show_ylabels=False)
plt.title('Asignacion de instancias', fontsize=11)
plt.subplot(323)
plot_decision_boundaries(kmeans_iter1, X, show_centroids=False, show_xlabels=False)
plot_centroids(kmeans_iter2.cluster_centers_)
plt.title('Reculculo de centroides', fontsize=11)
plt.subplot(324)
plot_decision_boundaries(kmeans_iter2, X, show_xlabels=False, show_ylabels=False)
plt.title('Nueva asignacion', fontsize=11)
plt.subplot(325)
plot_decision_boundaries(kmeans_iter2, X, show_centroids=False)
plot_centroids(kmeans_iter3.cluster_centers_)
plt.title('Centroides actualizados', fontsize=11)
plt.subplot(326)
plot_decision_boundaries(kmeans_iter3, X, show_ylabels=False)
plt.title('Asignacion final', fontsize=11)
plt.suptitle('Proceso iterativo de KMeans', fontsize=14)
plt.tight_layout()
plt.show()


### 1.8 — Impacto de la Inicialización Aleatoria


In [ ]:
# Función del docente preservada
def plot_clusterer_comparison(clusterer1, clusterer2, X, title1=None, title2=None):
    clusterer1.fit(X)
    clusterer2.fit(X)
    plt.figure(figsize=(12, 4))
    plt.subplot(121)
    plot_decision_boundaries(clusterer1, X)
    if title1:
        plt.title(title1, fontsize=13)
    plt.subplot(122)
    plot_decision_boundaries(clusterer2, X, show_ylabels=False)
    if title2:
        plt.title(title2, fontsize=13)

kmeans_rnd_init1 = KMeans(n_clusters=k, init='random', n_init=1,
                          algorithm='elkan', random_state=19)
kmeans_rnd_init2 = KMeans(n_clusters=k, init='k-means++', n_init=1,
                          algorithm='lloyd', random_state=19)
plot_clusterer_comparison(kmeans_rnd_init1, kmeans_rnd_init2, X,
                          'Solucion con init=random', 'Solucion con init=k-means++')
plt.tight_layout()
plt.show()
print('⚠️  La inicializacion aleatoria puede dar resultados suboptimos')
print('✅  k-means++ garantiza mejor convergencia')


In [ ]:
kmeans_rnd_10_inits = KMeans(n_clusters=k, init='random', n_init=10,
                              algorithm='elkan', random_state=11)
kmeans_rnd_10_inits.fit(X)
plt.figure(figsize=(9, 5))
plot_decision_boundaries(kmeans_rnd_10_inits, X)
plt.title('KMeans con n_init=10 inicializaciones aleatorias (elige la mejor)', fontsize=13)
plt.tight_layout()
plt.show()
print('✅ Con n_init=10 el modelo elige la mejor de 10 inicializaciones')


### 1.9 — Método del Codo (Elbow Method)

Graficamos la **inercia** (WCSS) vs número de clusters k. El 'codo' de la curva indica el k a partir del cual añadir más clusters no reduce significativamente la inercia.


In [ ]:
# {#yo: INERCIA (WCSS = Within-Cluster Sum of Squares):
#   Suma de distancias cuadradas de cada muestra a su centroide más cercano.
#   k=1 -> inercia máxima | k=m -> inercia=0 (cada punto es su propio cluster).
#   El codo es el punto donde la curva 'dobla': reducción marginal de inercia.
#   Se accede con kmeans.inertia_ tras el fit().
#   IMPORTANTE: combinar siempre con Silhouette Score porque el codo no siempre
#   es obvio en datos reales.}

k_min = 1
k_max = min(20, n_centers + 5)

print(f'⏳ Calculando inercia para k={k_min} hasta {k_max}...')
inercias = []
rango_k = range(k_min, k_max + 1)
for ki in rango_k:
    km = KMeans(n_clusters=ki, random_state=42, n_init=10)
    km.fit(X)
    inercias.append(km.inertia_)
    print(f'   k={ki:2d} -> inercia={km.inertia_:,.1f}')

# Deteccion automatica del codo (segunda derivada)
diferencias2 = np.diff(np.diff(inercias))
codo_idx = np.argmax(diferencias2) + 2
k_optimo_codo = list(rango_k)[codo_idx]
print(f'\n📍 Codo detectado en k = {k_optimo_codo}')

plt.figure(figsize=(10, 5))
plt.plot(rango_k, inercias, 'bo-', linewidth=2, markersize=7)
plt.axvline(x=k_optimo_codo, color='red', linestyle='--', linewidth=2,
            label=f'Codo detectado en k={k_optimo_codo}')
plt.axvline(x=n_centers, color='green', linestyle=':', linewidth=2,
            label=f'k real generado = {n_centers}')
plt.xlabel('Numero de clusters k', fontsize=13)
plt.ylabel('Inercia (WCSS)', fontsize=13)
plt.title('Metodo del Codo — Inercia vs Numero de Clusters', fontsize=14)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()
print(f'✅ Codo: k_opt={k_optimo_codo} | k_real={n_centers}')


### 1.10 — Silhouette Score: Valor Numérico

**Formula:** (b − a) / max(a, b)

- **a** = distancia media intra-cluster (cohesion)
- **b** = distancia media al cluster vecino mas cercano (separacion)

**Rango:** −1 a +1. Mas cercano a 1 = mejor separacion.


In [ ]:
from sklearn.metrics import silhouette_score

# {#yo: silhouette_score() calcula el promedio de todos los coeficientes de silueta.
#   Fórmula por muestra: (b - a) / max(a, b)
#     a = dist. promedio a otros puntos del MISMO cluster (cohesion interna)
#     b = dist. promedio al cluster VECINO más cercano (separacion externa)
#   Valor ~1  : muestra bien asignada, cluster compacto y separado
#   Valor ~0  : muestra en la frontera entre dos clusters
#   Valor negativo : muestra probablemente mal asignada
#   REQUIERE k >= 2. No tiene sentido con un solo cluster.}

score_actual = silhouette_score(X, kmeans.labels_)
print(f'✅ Silhouette Score (k={k}): {score_actual:.4f}')

print(f'\n⏳ Calculando Silhouette para k=2 hasta {k_max}...')
kmeans_por_k = [KMeans(n_clusters=ki, random_state=42, n_init=10).fit(X)
                for ki in range(1, k_max + 1)]
silhouette_scores = [silhouette_score(X, modelo.labels_)
                     for modelo in kmeans_por_k[1:]]
ks = list(range(2, k_max + 1))
k_optimo_sil = ks[np.argmax(silhouette_scores)]
print(f'📍 Silhouette maximo en k={k_optimo_sil}: {max(silhouette_scores):.4f}')

plt.figure(figsize=(10, 5))
plt.plot(ks, silhouette_scores, 'bo-', linewidth=2, markersize=7)
plt.axvline(x=k_optimo_sil, color='red', linestyle='--', linewidth=2,
            label=f'k optimo (Silhouette) = {k_optimo_sil}')
plt.axvline(x=n_centers, color='green', linestyle=':', linewidth=2,
            label=f'k real generado = {n_centers}')
plt.xlabel('Numero de clusters k', fontsize=13)
plt.ylabel('Silhouette Score', fontsize=13)
plt.title('Silhouette Score vs Numero de Clusters', fontsize=14)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()


### 1.11 — Diagramas de Silueta (4 valores de k)

La linea roja punteada = Silhouette Score promedio del modelo.


In [ ]:
from sklearn.metrics import silhouette_samples
from matplotlib.ticker import FixedLocator, FixedFormatter

# {#yo: silhouette_samples() devuelve el coeficiente de silueta INDIVIDUAL por muestra.
#   Permite ver qué clusters son compactos (barras largas a la derecha)
#   y cuáles tienen muestras mal asignadas (barras cortas o valores negativos).
#   Cluster ideal: todas las barras mas largas que la linea promedio, sin negativos.
#   'Forma de cuchilla' -> buena separacion entre clusters.}

ks_silueta = sorted(set([
    max(2, k_optimo_codo - 1),
    k_optimo_codo,
    k_optimo_sil,
    min(k_max, k_optimo_sil + 1)
]))[:4]
print(f'📊 Diagramas de silueta para k = {ks_silueta}')

plt.figure(figsize=(14, 10))
for plot_idx, ki in enumerate(ks_silueta):
    plt.subplot(2, 2, plot_idx + 1)
    y_pred_ki = kmeans_por_k[ki - 1].labels_
    coeficientes = silhouette_samples(X, y_pred_ki)
    padding = len(X) // 30
    pos = padding
    ticks = []
    for i in range(ki):
        coeffs = coeficientes[y_pred_ki == i]
        coeffs.sort()
        color = mpl.cm.Spectral(i / ki)
        plt.fill_betweenx(np.arange(pos, pos + len(coeffs)), 0, coeffs,
                          facecolor=color, edgecolor=color, alpha=0.7)
        ticks.append(pos + len(coeffs) // 2)
        pos += len(coeffs) + padding
    plt.gca().yaxis.set_major_locator(FixedLocator(ticks))
    plt.gca().yaxis.set_major_formatter(FixedFormatter(range(ki)))
    plt.ylabel('Cluster', fontsize=11)
    plt.gca().set_xticks([-0.1, 0, 0.2, 0.4, 0.6, 0.8, 1])
    plt.xlabel('Coeficiente de Silueta', fontsize=11)
    if ki - 2 < len(silhouette_scores):
        plt.axvline(x=silhouette_scores[ki - 2], color='red', linestyle='--', linewidth=2)
    plt.title(f'k={ki} | Score={silhouette_scores[ki-2]:.3f}', fontsize=13)
plt.suptitle('Diagramas de Silueta — Comparativa de k', fontsize=14)
plt.tight_layout()
plt.show()


### 1.12 — Diagramas de Silueta para TODOS los k


In [ ]:
n_ks = k_max - 1
cols = 2
filas = (n_ks + 1) // cols
plt.figure(figsize=(14, 4 * filas))
for plot_idx, ki in enumerate(range(2, k_max + 1)):
    plt.subplot(filas, cols, plot_idx + 1)
    y_pred_ki = kmeans_por_k[ki - 1].labels_
    coeficientes = silhouette_samples(X, y_pred_ki)
    padding = len(X) // 30
    pos = padding
    ticks = []
    for i in range(ki):
        coeffs = coeficientes[y_pred_ki == i]
        coeffs.sort()
        color = mpl.cm.Spectral(i / ki)
        plt.fill_betweenx(np.arange(pos, pos + len(coeffs)), 0, coeffs,
                          facecolor=color, edgecolor=color, alpha=0.7)
        ticks.append(pos + len(coeffs) // 2)
        pos += len(coeffs) + padding
    plt.gca().yaxis.set_major_locator(FixedLocator(ticks))
    plt.gca().yaxis.set_major_formatter(FixedFormatter(range(ki)))
    plt.ylabel('Cluster', fontsize=9)
    plt.gca().set_xticks([-0.1, 0, 0.2, 0.4, 0.6, 0.8, 1])
    plt.xlabel('Coeficiente de Silueta', fontsize=9)
    if ki - 2 < len(silhouette_scores):
        plt.axvline(x=silhouette_scores[ki - 2], color='red', linestyle='--')
    plt.title(f'k={ki} | Score={silhouette_scores[ki-2]:.3f}', fontsize=11)
plt.suptitle(f'Diagramas de Silueta completos — k de 2 a {k_max}', fontsize=14)
plt.tight_layout()
plt.show()
print(f'\n✅ RESUMEN SECCION 1:')
print(f'   Dataset con k={n_centers} clusters aleatorios')
print(f'   Metodo del Codo -> k_optimo = {k_optimo_codo}')
print(f'   Silhouette Score -> k_optimo = {k_optimo_sil}')
print(f'   Silhouette del modelo principal (k={k}): {score_actual:.4f}')


---
## 🌊 Sección 2 — Punto 2: Dataset Real — Carga y Exploración

**Sea Animals Image Dataset** (Kaggle — vencerlanz09):
- 23 clases de animales marinos
- ~11.500 imagenes (~500 por clase)
- Sin etiquetas para el clustering (las ignoramos en aprendizaje no supervisado)


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.preprocessing import StandardScaler

RUTA_DATASET  = '/content/drive/MyDrive/ML_Lab_NoSupervisado/Sea Animals'
TAMANO_IMG    = (64, 64)   # 64x64 px -> 4096 features por imagen
MODO_COLOR    = 'L'        # escala de grises (reduce dimension 3x vs RGB)
MAX_POR_CLASE = 500

CLASES = [
    'Clams','Corals','Crab','Dolphin','Eel','Fish',
    'Jelly Fish','Lobster','Nudibranchs','Octopus','Penguin',
    'Puffers','Rays','Seahorse','Sea Otter','Sea Urchins',
    'Seal','Shark','Shrimp','Squid','Starfish','Turtle','Whale'
]
N_CLASES = len(CLASES)

print(f'📁 Ruta: {RUTA_DATASET}')
print(f'🖼️  Tamano: {TAMANO_IMG[0]}x{TAMANO_IMG[1]} px')
print(f'📊 Features por imagen: {TAMANO_IMG[0] * TAMANO_IMG[1]}')
print(f'🐠 Numero de clases: {N_CLASES}')


In [ ]:
def cargar_dataset(ruta_base, clases, tamano=(64,64), modo='L', max_por_clase=500):
    """
    Carga imagenes del dataset Sea Animals desde Google Drive.
    Retorna:
        X_raw          : np.ndarray (m, n_features) normalizado [0,1]
        y_raw          : np.ndarray (m,) etiquetas numericas
        conteo_clases  : dict con imagenes cargadas por clase
    """
    # {#yo: Pipeline de carga y normalizacion:
    #   1. PIL.Image.open()  -> abre la imagen en memoria
    #   2. .convert('L')     -> convierte a escala de grises (1 canal en vez de 3)
    #   3. .resize(tamano)   -> estandariza dimension a 64x64
    #   4. np.array() / 255  -> NORMALIZACION al rango [0,1]
    #      Crucial porque KMeans usa distancias euclidianas:
    #      sin normalizar, pixeles brillantes dominan la distancia.
    #   5. .flatten()        -> convierte imagen 64x64 en vector de 4096 numeros}

    X_lista, y_lista = [], []
    conteo = {}
    for idx_clase, nombre in enumerate(clases):
        ruta_clase = os.path.join(ruta_base, nombre)
        if not os.path.isdir(ruta_clase):
            print(f'  ⚠️  Carpeta no encontrada: {nombre}')
            continue
        archivos = [f for f in os.listdir(ruta_clase)
                    if f.lower().endswith(('.jpg','.jpeg','.png','.webp'))][:max_por_clase]
        cargadas = 0
        for archivo in archivos:
            try:
                img = Image.open(os.path.join(ruta_clase, archivo)).convert(modo).resize(tamano)
                X_lista.append(np.array(img, dtype=np.float32).flatten() / 255.0)
                y_lista.append(idx_clase)
                cargadas += 1
            except Exception:
                continue
        conteo[nombre] = cargadas
        print(f'  ✅ {nombre}: {cargadas} imagenes')
    X_raw = np.array(X_lista)
    y_raw = np.array(y_lista)
    print(f'\n📊 Total: {X_raw.shape[0]} imagenes x {X_raw.shape[1]} features')
    return X_raw, y_raw, conteo

print('⏳ Cargando dataset... (puede tardar 2-4 minutos)')
X_imagenes, y_etiquetas, conteo_clases = cargar_dataset(
    RUTA_DATASET, CLASES, tamano=TAMANO_IMG, modo=MODO_COLOR, max_por_clase=MAX_POR_CLASE
)
print(f'\n✅ Carga completa | X: {X_imagenes.shape} | y: {y_etiquetas.shape}')


### 2.1 — Visualización de Muestras por Clase


In [ ]:
# 5 imagenes de cada una de las primeras 5 clases
n_mostrar = 5
clases_mostrar = min(5, N_CLASES)
fig, axes = plt.subplots(clases_mostrar, n_mostrar, figsize=(n_mostrar*2.5, clases_mostrar*2.5))
for fila, idx_clase in enumerate(range(clases_mostrar)):
    indices = np.where(y_etiquetas == idx_clase)[0][:n_mostrar]
    for col, idx in enumerate(indices):
        axes[fila, col].imshow(X_imagenes[idx].reshape(TAMANO_IMG), cmap='gray')
        axes[fila, col].axis('off')
        if col == 0:
            axes[fila, col].set_ylabel(CLASES[idx_clase], fontsize=9,
                                       rotation=0, labelpad=60, va='center')
plt.suptitle('Muestras del dataset Sea Animals (grises, 64x64)', fontsize=13)
plt.tight_layout()
plt.show()


In [ ]:
# Una imagen representativa de CADA una de las 23 clases
fig, axes = plt.subplots(4, 6, figsize=(16, 12))
axes = axes.flatten()
for idx_clase, nombre in enumerate(CLASES):
    indices = np.where(y_etiquetas == idx_clase)[0]
    if len(indices) == 0:
        axes[idx_clase].axis('off')
        continue
    axes[idx_clase].imshow(X_imagenes[indices[0]].reshape(TAMANO_IMG), cmap='gray')
    axes[idx_clase].set_title(nombre, fontsize=9)
    axes[idx_clase].axis('off')
for j in range(idx_clase + 1, len(axes)):
    axes[j].axis('off')
plt.suptitle('Una muestra por clase — 23 clases de animales marinos', fontsize=13)
plt.tight_layout()
plt.show()


### 2.2 — Análisis de Desbalance de Clases


In [ ]:
nombres_ord = list(conteo_clases.keys())
conteos     = [conteo_clases[n] for n in nombres_ord]
media    = np.mean(conteos)
maximo   = max(conteos)
minimo   = min(conteos)
ratio    = maximo / minimo if minimo > 0 else float('inf')

print('=' * 50)
print('ANALISIS DE DESBALANCE DE CLASES')
print('=' * 50)
for nombre, cnt in zip(nombres_ord, conteos):
    barra = '#' * int(cnt / maximo * 30)
    print(f'  {nombre:<15}: {cnt:4d} {barra}')
print(f'\n  Media  : {media:.0f} | Min: {minimo} | Max: {maximo} | Ratio: {ratio:.2f}x')
if ratio > 2:
    print('  ⚠️  Desbalance SIGNIFICATIVO -> usar class_weight=balanced')
else:
    print('  ✅ Dataset relativamente balanceado')

colores = ['#e74c3c' if c < media*0.8 else '#2ecc71' if c > media*1.2 else '#3498db'
           for c in conteos]
plt.figure(figsize=(14, 6))
barras = plt.bar(nombres_ord, conteos, color=colores, edgecolor='white')
plt.axhline(y=media, color='orange', linestyle='--', linewidth=2, label=f'Media={media:.0f}')
plt.xlabel('Clase', fontsize=12)
plt.ylabel('Numero de imagenes', fontsize=12)
plt.title('Distribucion de imagenes por clase — Sea Animals Dataset', fontsize=14)
plt.xticks(rotation=45, ha='right', fontsize=9)
plt.legend(fontsize=11)
plt.grid(axis='y', alpha=0.4)
for b, cnt in zip(barras, conteos):
    plt.text(b.get_x() + b.get_width()/2, b.get_height() + 2,
             str(cnt), ha='center', va='bottom', fontsize=7)
plt.tight_layout()
plt.show()


### 2.3 — Preprocesamiento: Normalización + PCA

Con 4.096 features por imagen, aplicamos **PCA** para reducir dimensionalidad antes de KMeans. Esto mejora la eficiencia y la calidad del clustering.


In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# {#yo: Preprocesamiento en dos pasos:
#   1. StandardScaler: normaliza features a media=0, std=1.
#      Aunque los pixeles ya estan en [0,1], estandarizar es necesario
#      para que PCA no este sesgado por diferencias de escala entre features.
#   2. PCA (Principal Component Analysis):
#      Reduce de 4096 features a N componentes que explican el 95% de varianza.
#      Por que PCA antes de KMeans:
#      - Maldicion de la dimensionalidad: en 4096D las distancias euclidianas
#        tienden a ser iguales, haciendo KMeans menos efectivo.
#      - PCA elimina ruido (features de baja varianza) -> clustering mas limpio.
#      - Acelera el entrenamiento: calcular distancias en ~150D vs 4096D.
#      n_components=0.95 -> mantiene los componentes que explican el 95% de varianza.}

print('⏳ Aplicando StandardScaler...')
escalador = StandardScaler()
X_escalado = escalador.fit_transform(X_imagenes)
print(f'✅ Escalado: media={X_escalado.mean():.4f}, std={X_escalado.std():.4f}')

print('\n⏳ Aplicando PCA (puede tardar 1-2 minutos)...')
pca = PCA(n_components=0.95, random_state=42)
X_pca = pca.fit_transform(X_escalado)
n_componentes       = pca.n_components_
varianza_acumulada  = float(np.sum(pca.explained_variance_ratio_))

print(f'✅ PCA completado:')
print(f'   Features originales : {X_imagenes.shape[1]}')
print(f'   Componentes PCA     : {n_componentes}')
print(f'   Varianza explicada  : {varianza_acumulada*100:.1f}%')
print(f'   Reduccion           : {(1 - n_componentes/X_imagenes.shape[1])*100:.1f}%')

varianza_cum = np.cumsum(pca.explained_variance_ratio_)
plt.figure(figsize=(10, 5))
plt.plot(range(1, n_componentes + 1), varianza_cum, 'b-', linewidth=2)
plt.axhline(y=0.95, color='red', linestyle='--', label='95% varianza')
plt.xlabel('Numero de componentes PCA', fontsize=12)
plt.ylabel('Varianza acumulada explicada', fontsize=12)
plt.title('PCA — Varianza acumulada vs Numero de componentes', fontsize=13)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()

X_procesado = X_pca
print(f'\n✅ X_procesado listo: {X_procesado.shape}  (imagenes x componentes_PCA)')


---
## 🔬 Sección 3 — Punto 2: KMeans + Elbow + Silhouette sobre Dataset Real

Usamos **MiniBatchKMeans** (mas eficiente para ~11.500 imagenes) y determinamos el k optimo con Metodo del Codo y Silhouette Score.


In [ ]:
from sklearn.cluster import MiniBatchKMeans
from sklearn.metrics import silhouette_score
import time

# {#yo: Por que MiniBatchKMeans y no KMeans normal?
#   KMeans normal procesa TODAS las muestras en cada iteracion -> lento con m>10.000.
#   MiniBatchKMeans usa mini-lotes (subconjuntos aleatorios) -> 10-50x mas rapido.
#   Parametros:
#   - batch_size: tamano del mini-lote (mayor = mas preciso, mas lento)
#   - init_size : tamano del conjunto de inicializacion
#   - n_init    : numero de inicializaciones (se queda con la mejor)
#   La precision es ligeramente inferior a KMeans pero aceptable para exploracion.}

K_RANGO = range(5, 31)
inercias_real    = []
silhouettes_real = []

print('⏳ Calculando Codo y Silhouette para dataset real...')
print('   (puede tardar 5-10 minutos)\n')
for ki in K_RANGO:
    t0 = time.time()
    mb = MiniBatchKMeans(n_clusters=ki, batch_size=1024,
                         init_size=3*ki, random_state=42, n_init=5)
    mb.fit(X_procesado)
    inercias_real.append(mb.inertia_)
    idx_m = np.random.choice(len(X_procesado), min(2000, len(X_procesado)), replace=False)
    sil = silhouette_score(X_procesado[idx_m], mb.labels_[idx_m])
    silhouettes_real.append(sil)
    print(f'  k={ki:2d} | Inercia={mb.inertia_:,.0f} | Silhouette={sil:.4f} | {time.time()-t0:.1f}s')

dif2 = np.diff(np.diff(inercias_real))
k_opt_codo_real = list(K_RANGO)[np.argmax(dif2) + 2]
k_opt_sil_real  = list(K_RANGO)[np.argmax(silhouettes_real)]
print(f'\n📍 Codo detectado en k = {k_opt_codo_real}')
print(f'📍 Silhouette maximo en k = {k_opt_sil_real}: {max(silhouettes_real):.4f}')


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
ax1.plot(K_RANGO, inercias_real, 'bo-', linewidth=2, markersize=7)
ax1.axvline(x=k_opt_codo_real, color='red', linestyle='--', linewidth=2,
            label=f'Codo en k={k_opt_codo_real}')
ax1.axvline(x=N_CLASES, color='green', linestyle=':', linewidth=2,
            label=f'Clases reales = {N_CLASES}')
ax1.set_xlabel('Numero de clusters k', fontsize=13)
ax1.set_ylabel('Inercia', fontsize=13)
ax1.set_title('Metodo del Codo — Sea Animals', fontsize=14)
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.4)
ax2.plot(K_RANGO, silhouettes_real, 'rs-', linewidth=2, markersize=7)
ax2.axvline(x=k_opt_sil_real, color='red', linestyle='--', linewidth=2,
            label=f'Optimo en k={k_opt_sil_real}')
ax2.axvline(x=N_CLASES, color='green', linestyle=':', linewidth=2,
            label=f'Clases reales = {N_CLASES}')
ax2.set_xlabel('Numero de clusters k', fontsize=13)
ax2.set_ylabel('Silhouette Score', fontsize=13)
ax2.set_title('Silhouette Score — Sea Animals', fontsize=14)
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.4)
plt.suptitle('Determinacion del k optimo — Sea Animals Dataset', fontsize=14)
plt.tight_layout()
plt.show()
print(f'\n✅ RESUMEN SECCION 3:')
print(f'   Metodo del Codo -> k={k_opt_codo_real}')
print(f'   Silhouette Score -> k={k_opt_sil_real}')
print(f'   K final para Seccion 4: k={k_opt_sil_real}')


In [ ]:
from sklearn.metrics import silhouette_samples
from matplotlib.ticker import FixedLocator, FixedFormatter

K_FINAL = k_opt_sil_real
print(f'⏳ Entrenando MiniBatchKMeans final con k={K_FINAL}...')
kmeans_real = MiniBatchKMeans(n_clusters=K_FINAL, batch_size=1024,
                               init_size=3*K_FINAL, random_state=42, n_init=10)
kmeans_real.fit(X_procesado)
print(f'✅ Modelo entrenado | Inercia = {kmeans_real.inertia_:,.0f}')

idx_m = np.random.choice(len(X_procesado), 3000, replace=False)
X_m = X_procesado[idx_m]
labels_m = kmeans_real.labels_[idx_m]
coefs = silhouette_samples(X_m, labels_m)

plt.figure(figsize=(12, 7))
padding = len(X_m) // 30
pos = padding
ticks = []
for i in range(K_FINAL):
    c = coefs[labels_m == i]
    c.sort()
    color = mpl.cm.tab20(i / K_FINAL)
    plt.fill_betweenx(np.arange(pos, pos + len(c)), 0, c,
                      facecolor=color, edgecolor=color, alpha=0.7)
    ticks.append(pos + len(c) // 2)
    pos += len(c) + padding
plt.gca().yaxis.set_major_locator(FixedLocator(ticks))
plt.gca().yaxis.set_major_formatter(FixedFormatter(range(K_FINAL)))
plt.ylabel('Cluster', fontsize=12)
plt.xlabel('Coeficiente de Silueta', fontsize=12)
score_km_real = silhouette_score(X_m, labels_m)
plt.axvline(x=score_km_real, color='red', linestyle='--', linewidth=2,
            label=f'Score promedio = {score_km_real:.3f}')
plt.title(f'Diagrama de Silueta — k={K_FINAL} (k optimo) | Sea Animals', fontsize=13)
plt.legend(fontsize=11)
plt.tight_layout()
plt.show()


---
## 🤖 Sección 4 — Punto 2: Aprendizaje Semi-Supervisado

Escenario: **~11.500 imagenes sin etiquetar**.

1. Entrenamos KMeans sobre los datos
2. Encontramos las imagenes **mas representativas** (mas cercanas a centroides)
3. **Simulamos etiquetado manual** de solo esas representativas
4. **Propagamos** etiquetas al resto del dataset
5. Entrenamos clasificador y evaluamos accuracy


In [ ]:
from sklearn.model_selection import train_test_split

X_train_raw, X_test_raw, y_train_real, y_test_real = train_test_split(
    X_procesado, y_etiquetas, test_size=0.25, random_state=42, stratify=y_etiquetas
)
print(f'✅ Division del dataset:')
print(f'   Entrenamiento: {X_train_raw.shape[0]} imagenes')
print(f'   Test         : {X_test_raw.shape[0]} imagenes')


In [ ]:
print(f'⏳ Entrenando MiniBatchKMeans con k={K_FINAL} sobre train set...')
kmeans_semi = MiniBatchKMeans(n_clusters=K_FINAL, batch_size=1024,
                               init_size=3*K_FINAL, random_state=42, n_init=10)

# {#yo: fit_transform() = fit() + transform() en un solo paso.
#   Retorna matriz de DISTANCIAS (m_train, K_FINAL):
#   fila i = imagen i, columna j = distancia al centroide del cluster j.
#   Luego argmin(axis=0) da la imagen MAS CERCANA a cada centroide -> representativa.}
X_distancias = kmeans_semi.fit_transform(X_train_raw)
print(f'✅ KMeans entrenado | distancias shape: {X_distancias.shape}')


In [ ]:
# {#yo: argmin(axis=0) opera sobre COLUMNAS.
#   Para cada cluster j, encuentra el indice de la imagen con MENOR distancia al centroide j.
#   Esa imagen es la 'mas representativa' del cluster.
#   Resultado: vector de K_FINAL indices (uno por cluster).}
indices_representativos  = np.argmin(X_distancias, axis=0)
X_representativas        = X_train_raw[indices_representativos]
y_representativas        = y_train_real[indices_representativos]

print(f'✅ Imagenes representativas encontradas: {len(indices_representativos)}')
print(f'   Una por cluster ({K_FINAL} clusters)')
print(f'   Clases representadas: {len(np.unique(y_representativas))}/{N_CLASES}')
print('\n📋 Distribucion de clases en representativas:')
for clase_id in np.unique(y_representativas):
    cnt = np.sum(y_representativas == clase_id)
    print(f'   {CLASES[clase_id]}: {cnt}')


In [ ]:
n_vis = min(20, K_FINAL)
cols_r = 5
filas_r = (n_vis + cols_r - 1) // cols_r
plt.figure(figsize=(14, filas_r * 3))
for i in range(n_vis):
    idx_orig = indices_representativos[i]
    img_vec  = X_imagenes[idx_orig].reshape(TAMANO_IMG)
    plt.subplot(filas_r, cols_r, i + 1)
    plt.imshow(img_vec, cmap='gray')
    plt.title(f'Cluster {i}\n{CLASES[y_representativas[i]]}', fontsize=8)
    plt.axis('off')
plt.suptitle(f'Primeras {n_vis} imagenes mas representativas (mas cercanas a centroides)', fontsize=13)
plt.tight_layout()
plt.show()
print('🔍 Estas son las imagenes que un experto etiquetaria manualmente')


In [ ]:
# {#yo: Propagacion de etiquetas (Label Propagation con KMeans):
#   Principio: si la imagen representativa del cluster j tiene clase C,
#   TODAS las imagenes del cluster j reciben la clase C.
#   Es propagacion basada en proximidad geometrica.
#   Ventaja: etiquetamos miles de imagenes habiendo revisado solo K manualmente.
#   Desventaja: introduce ruido — no todas las imagenes de un cluster son del mismo animal.
#   Cuanto mejor el clustering (Silhouette alto), menor el ruido.}

y_train_propagado = np.empty(len(X_train_raw), dtype=int)
for i in range(K_FINAL):
    mascara = kmeans_semi.labels_ == i
    y_train_propagado[mascara] = y_representativas[i]

print('✅ Etiquetas propagadas a todo el conjunto de entrenamiento')
print(f'   {len(y_train_propagado)} imagenes etiquetadas usando {K_FINAL} etiquetas manuales')


In [ ]:
from sklearn.linear_model import LogisticRegression

# {#yo: class_weight='balanced' — CRITICO para datasets desbalanceados.
#   Sin 'balanced': el clasificador predice siempre la clase mas frecuente,
#   ignorando las clases minoritarias.
#   Con 'balanced': pesos w_k = n_total / (n_clases * n_muestras_k)
#   -> clases con pocas muestras reciben mayor 'importancia' en el entrenamiento.
#   Especialmente importante aqui porque la propagacion puede crear desbalance.}

# Modelo A: Solo representativas
print('=' * 55)
print('MODELO A: Solo imagenes representativas')
clf_representativas = LogisticRegression(multi_class='ovr', solver='lbfgs',
                                          max_iter=5000, random_state=42,
                                          class_weight='balanced')
clf_representativas.fit(X_representativas, y_representativas)
acc_representativas = clf_representativas.score(X_test_raw, y_test_real)
print(f'✅ Accuracy con {K_FINAL} representativas: {acc_representativas:.4f} ({acc_representativas*100:.1f}%)')

# Modelo B: Etiquetas propagadas
print('\n' + '=' * 55)
print('MODELO B: Etiquetas propagadas (todo el training set)')
clf_propagado = LogisticRegression(multi_class='ovr', solver='lbfgs',
                                    max_iter=5000, random_state=42,
                                    class_weight='balanced')
clf_propagado.fit(X_train_raw, y_train_propagado)
acc_propagado = clf_propagado.score(X_test_raw, y_test_real)
print(f'✅ Accuracy con etiquetas propagadas: {acc_propagado:.4f} ({acc_propagado*100:.1f}%)')

# Modelo C: Baseline — 50 muestras aleatorias
print('\n' + '=' * 55)
print('MODELO C (Baseline): 50 muestras aleatorias')
idx_random = np.random.choice(len(X_train_raw), 50, replace=False)
clf_random = LogisticRegression(multi_class='ovr', solver='lbfgs',
                                 max_iter=5000, random_state=42,
                                 class_weight='balanced')
clf_random.fit(X_train_raw[idx_random], y_train_real[idx_random])
acc_random = clf_random.score(X_test_raw, y_test_real)
print(f'✅ Accuracy con 50 aleatorias: {acc_random:.4f} ({acc_random*100:.1f}%)')

print('\n' + '=' * 55)
print('COMPARATIVA SEMI-SUPERVISADO')
print('=' * 55)
print(f'  Baseline (50 random)                  : {acc_random*100:.1f}%')
print(f'  Representativas KMeans ({K_FINAL} imgs): {acc_representativas*100:.1f}%')
print(f'  Propagadas (todo train)                : {acc_propagado*100:.1f}%')
print(f'  Mejora vs baseline                     : {(acc_representativas-acc_random)*100:+.1f}%')


---
## 🎯 Sección 5 — Punto 2: Aprendizaje Activo

El **Aprendizaje Activo** mejora el modelo identificando las muestras donde el clasificador tiene **menor confianza** y corrigiendolas con etiquetas reales (simulando la revision de un experto).


In [ ]:
# {#yo: predict_proba() retorna la probabilidad de cada clase para cada muestra.
#   Shape: (m, n_clases). Cada fila suma 1.
#   Estrategia de menor confianza:
#   - Calculamos la probabilidad MAXIMA de cada muestra (la clase mas probable).
#   - Si esa prob. maxima es baja (ej. 0.08), el modelo no esta seguro de nada.
#   - Esas muestras inciertas son las candidatas a ser corregidas por el experto.
#   argmax(axis=1) -> clase predicha | max(axis=1) -> confianza de la prediccion}

print('⏳ Calculando probabilidades del clasificador propagado...')
probas     = clf_propagado.predict_proba(X_train_raw)
confianzas = np.max(probas, axis=1)
labels_pred = np.argmax(probas, axis=1)

print(f'✅ Probabilidades calculadas')
print(f'   Confianza promedio  : {confianzas.mean():.4f}')
print(f'   Confianza minima    : {confianzas.min():.4f}')
print(f'   Confianza maxima    : {confianzas.max():.4f}')
print(f'   Muestras < 10% conf : {np.sum(confianzas < 0.10)}')

plt.figure(figsize=(10, 5))
plt.hist(confianzas, bins=50, color='steelblue', edgecolor='white', alpha=0.8)
plt.axvline(x=np.percentile(confianzas, 10), color='red', linestyle='--', linewidth=2,
            label=f'Percentil 10 = {np.percentile(confianzas, 10):.3f}')
plt.xlabel('Confianza del modelo (probabilidad maxima)', fontsize=12)
plt.ylabel('Numero de muestras', fontsize=12)
plt.title('Distribucion de confianza — Identificar muestras inciertas', fontsize=13)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()


In [ ]:
N_ACTIVO = K_FINAL
indices_ordenados = np.argsort(confianzas)
indices_inciertos = indices_ordenados[:N_ACTIVO]

X_inciertos      = X_train_raw[indices_inciertos]
y_inciertos_pred = labels_pred[indices_inciertos]
y_inciertos_real = y_train_real[indices_inciertos]
conf_inciertos   = confianzas[indices_inciertos]

print(f'✅ {N_ACTIVO} muestras de menor confianza identificadas')
print(f'   Confianza promedio de inciertas: {conf_inciertos.mean():.4f}')
print(f'   Errores detectados: {np.sum(y_inciertos_pred != y_inciertos_real)}/{N_ACTIVO}')
print(f'   (errores = muestras mal etiquetadas por propagacion)')

n_vis2 = min(15, N_ACTIVO)
plt.figure(figsize=(15, 4))
for i in range(n_vis2):
    idx_orig = indices_inciertos[i]
    img_vec  = X_imagenes[idx_orig].reshape(TAMANO_IMG)
    plt.subplot(3, 5, i + 1)
    plt.imshow(img_vec, cmap='gray')
    pred_n = CLASES[y_inciertos_pred[i]]
    real_n = CLASES[y_inciertos_real[i]]
    color  = 'red' if y_inciertos_pred[i] != y_inciertos_real[i] else 'green'
    plt.title(f'P:{pred_n[:7]}\nR:{real_n[:7]}\n({conf_inciertos[i]:.2f})',
              fontsize=6, color=color)
    plt.axis('off')
plt.suptitle('Muestras de menor confianza (rojo=error, verde=correcto)', fontsize=12)
plt.tight_layout()
plt.show()


In [ ]:
# {#yo: En Aprendizaje Activo REAL:
#   Un experto humano revisa las muestras inciertas y corrige sus etiquetas.
#   Aqui lo SIMULAMOS usando las etiquetas reales del dataset.
#   Luego reemplazamos en el conjunto de entrenamiento y reentrenamos.
#   Logica: si mejoramos las etiquetas mas 'dificiles', el modelo aprende mejor
#   que si etiquetamos muestras aleatorias.}

y_train_activo = y_train_propagado.copy()
y_train_activo[indices_inciertos] = y_inciertos_real  # simulacion de corrección experto

n_cambios = int(np.sum(y_train_activo != y_train_propagado))
print(f'✅ Correccion simulada: {n_cambios} etiquetas actualizadas')

clf_activo = LogisticRegression(multi_class='ovr', solver='lbfgs',
                                 max_iter=5000, random_state=42,
                                 class_weight='balanced')
clf_activo.fit(X_train_raw, y_train_activo)
acc_activo = clf_activo.score(X_test_raw, y_test_real)

print(f'\n📊 COMPARATIVA APRENDIZAJE ACTIVO:')
print(f'   Antes (propagadas)                      : {acc_propagado*100:.1f}%')
print(f'   Despues (correccion activa {N_ACTIVO} muestras): {acc_activo*100:.1f}%')
print(f'   Mejora por Aprendizaje Activo            : {(acc_activo - acc_propagado)*100:+.1f}%')


In [ ]:
# Iteracion 2 de Aprendizaje Activo
print('⏳ Iteracion 2 de Aprendizaje Activo...')
probas2     = clf_activo.predict_proba(X_train_raw)
confianzas2 = np.max(probas2, axis=1)
indices_inciertos2 = np.argsort(confianzas2)[:N_ACTIVO]

y_train_activo2 = y_train_activo.copy()
y_train_activo2[indices_inciertos2] = y_train_real[indices_inciertos2]

clf_activo2 = LogisticRegression(multi_class='ovr', solver='lbfgs',
                                  max_iter=5000, random_state=42,
                                  class_weight='balanced')
clf_activo2.fit(X_train_raw, y_train_activo2)
acc_activo2 = clf_activo2.score(X_test_raw, y_test_real)

print(f'   Iteracion 2: {acc_activo2*100:.1f}%')
print(f'   Mejora adicional: {(acc_activo2 - acc_activo)*100:+.1f}%')

etapas   = ['Baseline\n(50 random)', 'Representativas\n(KMeans)',
            'Propagadas\n(todo train)', 'Activo\nIter 1', 'Activo\nIter 2']
accuracies = [acc_random, acc_representativas, acc_propagado, acc_activo, acc_activo2]
paleta   = ['#95a5a6', '#3498db', '#2ecc71', '#e67e22', '#e74c3c']

plt.figure(figsize=(12, 6))
barras = plt.bar(etapas, [a*100 for a in accuracies], color=paleta,
                 edgecolor='white', linewidth=1.5)
for b, acc in zip(barras, accuracies):
    plt.text(b.get_x() + b.get_width()/2, b.get_height() + 0.3,
             f'{acc*100:.1f}%', ha='center', va='bottom', fontsize=12, fontweight='bold')
plt.ylabel('Accuracy (%)', fontsize=13)
plt.title('Progresion de Accuracy — Semi-Supervisado + Aprendizaje Activo', fontsize=14)
plt.ylim(0, max(a*100 for a in accuracies) * 1.18)
plt.grid(axis='y', alpha=0.4)
plt.tight_layout()
plt.show()


---
## 📈 Sección 6 — Comparativa Final de Resultados


In [ ]:
import pandas as pd

resultados = {
    'Metodo': [
        'Baseline (50 muestras aleatorias)',
        'Semi-Supervisado: Representativas KMeans',
        'Semi-Supervisado: Etiquetas Propagadas',
        'Aprendizaje Activo — Iteracion 1',
        'Aprendizaje Activo — Iteracion 2'
    ],
    'Muestras etiquetadas': [
        '50 (aleatorias)',
        str(K_FINAL) + ' (representativas)',
        'Todas (propagadas)',
        'Todas + ' + str(N_ACTIVO) + ' corregidas',
        'Todas + ' + str(2*N_ACTIVO) + ' corregidas'
    ],
    'Accuracy': [
        f'{acc_random*100:.2f}%',
        f'{acc_representativas*100:.2f}%',
        f'{acc_propagado*100:.2f}%',
        f'{acc_activo*100:.2f}%',
        f'{acc_activo2*100:.2f}%'
    ],
    'Mejora vs Baseline': [
        '---',
        f'{(acc_representativas-acc_random)*100:+.2f}%',
        f'{(acc_propagado-acc_random)*100:+.2f}%',
        f'{(acc_activo-acc_random)*100:+.2f}%',
        f'{(acc_activo2-acc_random)*100:+.2f}%'
    ]
}
df = pd.DataFrame(resultados)
print('=' * 80)
print('TABLA COMPARATIVA FINAL — LABORATORIO APRENDIZAJE NO SUPERVISADO')
print('=' * 80)
print(df.to_string(index=False))
print('=' * 80)


In [ ]:
metodos  = ['Baseline\n(50 rand)', 'Repr. KMeans', 'Propagadas', 'Activo Iter1', 'Activo Iter2']
valores  = [acc_random*100, acc_representativas*100,
            acc_propagado*100, acc_activo*100, acc_activo2*100]
paleta2  = ['#bdc3c7', '#3498db', '#27ae60', '#f39c12', '#c0392b']

fig, ax = plt.subplots(figsize=(13, 7))
barras = ax.bar(metodos, valores, color=paleta2, edgecolor='white', linewidth=1.5, width=0.6)
for b, v in zip(barras, valores):
    ax.text(b.get_x() + b.get_width()/2, b.get_height() + 0.3,
            f'{v:.1f}%', ha='center', va='bottom', fontsize=13, fontweight='bold')
ax.axhline(y=acc_random*100, color='gray', linestyle='--', alpha=0.7, label='Baseline')
ax.set_ylabel('Accuracy (%)', fontsize=13)
ax.set_title('Comparativa Final de Metodos — Sea Animals (23 clases)', fontsize=14)
ax.set_ylim(0, max(valores) * 1.18)
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.4)
plt.tight_layout()
plt.show()

print('\n📝 CONCLUSIONES:')
print('1. El aprendizaje semi-supervisado con KMeans permite obtener')
print('   buenos clasificadores etiquetando manualmente muy pocas imagenes.')
print('2. Las imagenes representativas (cercanas al centroide) son mas')
print('   informativas que muestras aleatorias.')
print('3. La propagacion de etiquetas escala bien pero introduce ruido.')
print('4. El Aprendizaje Activo corrige las muestras mas problematicas,')
print('   mejorando el accuracy iterativamente.')
print('5. PCA fue esencial para hacer el clustering eficiente en imagenes.')


---
## 📚 Sección 7 — Glosario y Guía de Estudio para la Defensa

Estudia esta sección completa antes de la defensa oral.


### 📖 Glosario Completo

| Termino | Definicion simple | Formula (si aplica) | Para la defensa, di... |
|---|---|---|---|
| **Aprendizaje No Supervisado** | ML sin etiquetas: el modelo descubre estructura | — | *'Usamos datos sin etiquetar para encontrar patrones ocultos'* |
| **Clustering** | Agrupar muestras similares en grupos (clusters) | — | *'Agrupamos imagenes similares sin conocer las clases a priori'* |
| **KMeans** | Algoritmo iterativo: asigna cada punto al centroide mas cercano | min sum ||xi - mu_k||^2 | *'Minimiza la distancia intra-cluster eligiendo k centroides'* |
| **Centroide** | Punto promedio de todas las muestras de un cluster | mu_k = (1/n_k) * sum(xi) | *'El centro geometrico del cluster, se recalcula cada iteracion'* |
| **Inercia (WCSS)** | Suma de distancias cuadradas al centroide | sum ||xi - mu_k||^2 | *'Menor inercia = clusters mas compactos. Accedo con kmeans.inertia_'* |
| **Hard Clustering** | Cada muestra pertenece a UN solo cluster | labels_ en {0,...,k-1} | *'Asignacion binaria. Uso kmeans.labels_'* |
| **Soft Clustering** | Distancia de cada muestra a cada centroide | transform() -> distancias | *'Asignacion suave. Uso kmeans.transform()'* |
| **Metodo del Codo** | Grafica inercia vs k; el codo = k optimo | — | *'Busco el punto donde agregar mas k no reduce significativamente la inercia'* |
| **Silhouette Score** | Mide separacion de clusters | (b-a)/max(a,b) | *'Cercano a 1 = bien asignado, ~0 = en frontera, negativo = error'* |
| **a (silueta)** | Distancia media a puntos del MISMO cluster | a = media(dist a mismo cluster) | *'Cohesion: que tan dentro esta del cluster'* |
| **b (silueta)** | Distancia media al cluster VECINO mas cercano | b = min_k(media(dist al cluster k)) | *'Separacion: que tan lejos esta del cluster vecino'* |
| **k-means++** | Inicializacion inteligente de centroides | — | *'Distribuye centroides iniciales lejos entre si, mejor convergencia'* |
| **MiniBatchKMeans** | KMeans que usa mini-lotes (mas rapido) | — | *'Lo uso porque con >10.000 muestras KMeans normal es muy lento'* |
| **PCA** | Reduccion dimensional que maximiza varianza | X_pca = X * V_k | *'Reduje de 4096 a N componentes preservando el 95% de informacion'* |
| **Semi-Supervisado** | Aprender con pocas etiquetas y muchos datos sin etiquetar | — | *'Etiqueto solo las representativas (K) y propago al resto'* |
| **Imagen representativa** | La mas cercana al centroide del cluster | argmin distancia | *'La imagen mas tipica del cluster. Uso argmin(distancias, axis=0)'* |
| **Propagacion de etiquetas** | Asignar al cluster la etiqueta de su representativa | — | *'Todos los del cluster k reciben la clase de la imagen mas cercana al centroide'* |
| **Aprendizaje Activo** | Iterar etiquetando muestras de menor confianza | — | *'Identifico muestras inciertas con predict_proba y las corrijo primero'* |
| **predict_proba()** | Probabilidad de pertenencia a cada clase | — | *'Devuelve vector de probs por clase. El max es la confianza del modelo'* |
| **class_weight=balanced** | Penaliza mas los errores en clases minoritarias | w_k = n/(K*n_k) | *'Lo uso porque el dataset puede tener desbalance: evita ignorar clases pequenas'* |
| **StandardScaler** | Normaliza features a media=0, std=1 | z = (x-mu)/sigma | *'Estandarizo antes de PCA para que todas las features pesen igual'* |


### ❓ Preguntas Frecuentes de Defensa — Con Respuestas

---

**P1: Por que usas MiniBatchKMeans y no KMeans normal en el dataset real?**

> KMeans normal procesa TODAS las muestras en cada iteracion. Con ~11.500 imagenes y ~150 componentes PCA, cada iteracion es costosa. MiniBatchKMeans usa mini-lotes (subconjuntos de ~1024 muestras por paso), siendo 10-50x mas rapido. La perdida de precision es minima y aceptable para exploracion.

---

**P2: Que significa un Silhouette Score de 0?**

> La muestra esta en la frontera exacta entre dos clusters. La distancia a su cluster (a) es igual a la distancia al cluster vecino (b), dando (b-a)/max(a,b) = 0. Es una muestra ambigua.

---

**P3: Por que el Metodo del Codo y Silhouette no siempre dan el mismo k optimo?**

> Miden cosas distintas. El codo mide inercia (compacidad interna). El Silhouette mide compacidad + separacion entre clusters. Un k puede tener clusters compactos (codo) pero mal separados (bajo Silhouette). Siempre es mejor usar ambas metricas juntas.

---

**P4: Por que aplicaste PCA antes de KMeans?**

> Tres razones: (1) Maldicion de la dimensionalidad: en 4096D las distancias euclidianas tienden a converger, haciendo KMeans poco efectivo. (2) Eficiencia: distancias en 150D son mucho mas rapidas que en 4096D. (3) Ruido: PCA elimina features de baja varianza.

---

**P5: Que es k-means++ y por que es mejor que init=random?**

> k-means++ inicializa el primer centroide aleatoriamente y los siguientes con probabilidad proporcional a la distancia cuadrada al centroide mas cercano ya elegido. Esto garantiza centroides bien distribuidos, reduce iteraciones y evita minimos locales. Con 'random', todos los centroides pueden caer en la misma zona.

---

**P6: Como simulaste el etiquetado manual?**

> En la practica real, un experto humano veria las imagenes representativas y les asignaria una etiqueta. Como tenemos las etiquetas reales, usamos y_representativas = y_train_real[indices_representativos]. Esto simula el proceso perfectamente: el experto solo revisa K imagenes (una por cluster).

---

**P7: Por que a veces la propagacion empeora el modelo vs solo representativas?**

> Al propagar, etiquetamos TODAS las imagenes del cluster con la clase de su representativa. Pero no todas son del mismo animal: hay 'contaminacion' entre clases. Este ruido puede confundir al clasificador. Por eso el Aprendizaje Activo corrige primero las muestras donde el modelo detecta mayor incertidumbre (probablemente las mal etiquetadas).

---

**P8: Que hace argmin(X_distancias, axis=0) exactamente?**

> X_distancias tiene forma (m, K): filas=imagenes, columnas=clusters. argmin(axis=0) opera sobre cada COLUMNA: para cada cluster j, encuentra el INDICE de la fila (imagen) con MENOR distancia al centroide j. Resultado: vector de K indices, uno por cluster. Esa imagen es la mas representativa.

---

**P9: Por que usas class_weight=balanced?**

> Aunque cargamos ~500 imagenes por clase, la propagacion puede crear desbalance (algunos clusters capturan mas imagenes que otros). Sin 'balanced', el clasificador predice siempre la clase mas frecuente. 'balanced' asigna pesos inversamente proporcionales al tamano de la clase, compensando el desbalance.

---

**P10: Cual es la diferencia entre Aprendizaje Semi-Supervisado y Aprendizaje Activo?**

> Semi-supervisado: usamos KMeans para seleccionar CUALES muestras etiquetar (las representativas). Lo hacemos UNA vez antes de entrenar. Activo: el modelo YA ENTRENADO identifica ITERATIVAMENTE las muestras de menor confianza. Se etiquetan, se corrigen y se reentrena. Es mejora continua guiada por la incertidumbre del modelo, no por la geometria de los clusters.

---

**P11: Por que el Silhouette Score es bajo en imagenes reales vs datos sinteticos?**

> En datos sinteticos generamos clusters artificialmente bien separados. Las imagenes reales son visualmente complejas: un cangrejo y una langosta comparten features similares. La separabilidad en el espacio PCA de pixeles es mucho menor. Un Silhouette de 0.05-0.25 es normal para imagenes.

---

**P12: Cuantas imagenes etiqueto manualmente el experto?**

> Solo K_FINAL imagenes (una por cluster). De ~8.600 imagenes de entrenamiento, el experto revisa menos del 1%. Esto demuestra la potencia del aprendizaje semi-supervisado: maxima informacion con minimo esfuerzo de etiquetado.



### ⚠️ Errores Comunes y Como Evitarlos

**Error 1: Calcular Silhouette con k=1**
> ❌ silhouette_score con todos en un cluster -> error. ✅ Siempre empezar desde k=2.

---

**Error 2: No normalizar imagenes antes de KMeans**
> ❌ Usar pixeles en [0, 255] directamente. ✅ Dividir entre 255.0. KMeans usa distancias euclidianas; sin normalizar, pixeles brillantes dominan.

---

**Error 3: Aplicar PCA sin estandarizar primero**
> ❌ pca.fit(X_imagenes) directamente. ✅ StandardScaler().fit_transform(X) antes de PCA. Sin escalar, features de mayor varianza dominan los componentes.

---

**Error 4: Usar KMeans normal con mas de 10.000 muestras**
> ❌ KMeans(n_clusters=k).fit(X_grande) -> muy lento. ✅ MiniBatchKMeans(n_clusters=k, batch_size=1024) -> mucho mas eficiente.

---

**Error 5: Calcular Silhouette sobre todo el dataset con m > 10.000**
> ❌ silhouette_score(X_grande, labels) -> puede tardar horas (O(m^2)). ✅ Calcular sobre muestra: idx = np.random.choice(len(X), 2000); silhouette_score(X[idx], labels[idx]).

---

**Error 6: No usar class_weight=balanced con datos desbalanceados**
> ❌ LogisticRegression().fit(X, y) con clases desiguales -> modelo sesgado. ✅ LogisticRegression(class_weight='balanced').

---

**Error 7: Confundir fit_transform con fit + transform**
> ❌ kmeans.fit(X); dist = kmeans.transform(X) -> doble calculo. ✅ dist = kmeans.fit_transform(X) -> mas eficiente.

---

### ✅ Resumen Final

| Seccion | Tecnica | Dataset | Resultado clave |
|---|---|---|---|
| 1 | KMeans + Codo + Silhouette | Sintetico (make_blobs) | k optimo detectado automaticamente |
| 2 | Carga y exploracion | Sea Animals (23 clases) | ~11.500 imgs x 4.096 features |
| 3 | KMeans + PCA + Codo + Silhouette | Sea Animals | k optimo para imagenes reales |
| 4 | Aprendizaje Semi-Supervisado | Sea Animals | Accuracy con pocas etiquetas |
| 5 | Aprendizaje Activo | Sea Animals | Mejora iterativa del clasificador |
| 6 | Comparativa final | Todos | Tabla y grafica comparativa |

> 🎓 **Para la defensa:** domina los conceptos de Silhouette (a, b, formula), el flujo de semi-supervisado (KMeans -> representativas -> propagacion -> clasificador), y el flujo de activo (predict_proba -> menor confianza -> correccion -> reentrenar).
